# 5. Full English translation

The current 1,007 Ukrainian triplets were translated into English with
**GPT-6 Astra**. The [current dataset walkthrough](../../warbias-artifacts/notebooks/05_full_english_translation.ipynb)
shows those saved translations and verifies their text bindings. The ten-item
COMET sample in notebook 4 uses GPT-6.1 Sol and is separate from this release.

This notebook prepares translation for a new complete Ukrainian generation.
Leave the source paths unset until the generation and its manifest are available.
A configured provider factory is an alternative to recorded assistant output;
set its actual provider, model and settings in `translator` before execution.


In [1]:
import json
from pathlib import Path

from pandas import DataFrame, set_option

from fairforget.dataset_workflow import create_workspace, find_project
from fairforget.io import read_rows, write_rows
from fairforget.notebook_display import display

project = find_project()
protocol = json.loads((project / "protocol/construction.json").read_text())
work = create_workspace("05_full_english_translation")
set_option("display.max_colwidth", None)
set_option("display.max_rows", 40)

## New-run files and output contract

Use `triplets.jsonl` and `production-result.json` from a complete Ukrainian run.
Supply recorded English translations with the same IDs, or configure an actual
translation provider. Preserve all three sentence roles and keep both languages.
The default execution prepares this workflow without making model calls.


In [2]:
display(
    DataFrame(
        [
            {"file": "triplets.jsonl", "role": "Ukrainian source", "required": True},
            {
                "file": "production-result.json",
                "role": "Complete source generation manifest",
                "required": True,
            },
            {
                "file": "assistant-translations.jsonl",
                "role": "Recorded English text when using assistant capture",
                "required": False,
            },
            {
                "file": "translation/ukrainian.jsonl",
                "role": "Retained Ukrainian output",
                "required": False,
            },
            {
                "file": "translation/english.jsonl",
                "role": "Aligned English output",
                "required": False,
            },
        ]
    )
)

,file,role,required
0,triplets.jsonl,Ukrainian source,True
1,production-result.json,Complete source generation manifest,True
2,assistant-translations.jsonl,Recorded English text when using assistant capture,False
3,translation/ukrainian.jsonl,Retained Ukrainian output,False
4,translation/english.jsonl,Aligned English output,False


In [3]:
from fairforget.dataset_workflow import preview_translation, record_translations
from fairforget.translation.warbias import align_languages, validate_production_source

translator = json.loads((project / "configs/translation/astra.json").read_text())
UKRAINIAN_PATH = None
PRODUCTION_MANIFEST_PATH = None
ASSISTANT_TRANSLATIONS_PATH = None
PROVIDER_FACTORY = None
RUN_TRANSLATION = False

result = {
    "stage": "full_english_translation",
    "status": "waiting_for_ukrainian_input",
    "source_language": "uk",
    "target_language": "en",
    "translation_model": translator["model"],
    "annotation_status": protocol["full_ukrainian_annotation"]["status"],
    "execution_requested": RUN_TRANSLATION,
}
if UKRAINIAN_PATH is not None:
    if PRODUCTION_MANIFEST_PATH is None:
        raise ValueError("Supply the complete source generation manifest")
    ukrainian = read_rows(UKRAINIAN_PATH)
    production = json.loads(Path(PRODUCTION_MANIFEST_PATH).read_text())
    validate_production_source(ukrainian, production, translator)
    display(DataFrame(ukrainian)[["id", "stereotype", "counter_stereotype", "unrelated"]].head())
    result = {
        "stage": "full_english_translation",
        "translation_model": translator["model"],
        **preview_translation(
            project,
            ukrainian,
            work / "translation-preview",
            config=translator,
            production_manifest=production,
        ),
        "execution_requested": RUN_TRANSLATION,
    }
if RUN_TRANSLATION:
    if UKRAINIAN_PATH is None:
        raise ValueError("Supply the full Ukrainian input before execution")
    if ASSISTANT_TRANSLATIONS_PATH is not None:
        english = record_translations(
            project,
            ukrainian,
            read_rows(ASSISTANT_TRANSLATIONS_PATH),
            work / "translation",
            config=translator,
            production_manifest=production,
            source_origin={
                "kind": "full_ukrainian_generation",
                "manifest_sha256": production["manifest_sha256"],
            },
        )
    elif PROVIDER_FACTORY:
        from fairforget.cli_support import load_factory
        from fairforget.translation.warbias import translate_production

        provider = load_factory(PROVIDER_FACTORY)(config=translator)
        english = translate_production(
            ukrainian,
            production,
            translator,
            project / translator["prompt"],
            work / "translation",
            provider=provider,
            execute=True,
        )
    else:
        raise ValueError("Supply recorded assistant translations or a provider factory")
    align_languages(ukrainian, english)
    display(DataFrame(english)[["id", "stereotype", "counter_stereotype", "unrelated"]].head())
    write_rows(work / "translation/ukrainian.jsonl", ukrainian)
    write_rows(work / "translation/english.jsonl", english)
    result = {
        "stage": "full_english_translation",
        "status": "translated",
        "pairs": len(english),
        "translation_model": translator["model"],
    }
result

{'stage': 'full_english_translation',
 'status': 'waiting_for_ukrainian_input',
 'source_language': 'uk',
 'target_language': 'en',
 'translation_model': 'gpt-6-astra',
 'annotation_status': 'in_progress',
 'execution_requested': False}

Set the source paths to validate a new generation and prepare translation
requests. Set `RUN_TRANSLATION = True` to capture supplied assistant translations
or call the configured provider. Both aligned languages retain source bindings.
The result records the model configured for that run.
